In [30]:
import sys
sys.path.append("..")


In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from factor_engines.load_data import get_price_data, compute_returns
from factor_engines.factors import (
    momentum_factor,
    volatility_factor,
    reversal_factor,
    sma_distance_factor,
    compute_all,
    zscore,
    daily_score
)
from factor_engines.portfolio import (
    build_long_short_portfolio,
    build_zscore_portfolio_from_factor,
    build_composite_portfolio
)
from factor_engines.regression import (
    summarize_regression
)

plt.style.use("seaborn-v0_8")


In [32]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

prices = get_price_data(tickers, "2015-01-01", "2025-01-01")
returns = compute_returns(prices)


In [33]:
momentum = momentum_factor(prices)
volatility = volatility_factor(prices)
reversal = reversal_factor(prices)
sma = sma_distance_factor(prices)

factors = {
    "momentum": momentum,
    "volatility": volatility,
    "reversal": reversal,
    "sma_distance": sma
}


In [34]:
mom_weights_ls, mom_ret_ls = build_long_short_portfolio(momentum, returns)
mom_weights_z, mom_ret_z = build_zscore_portfolio_from_factor(momentum, returns)
comp_weights, comp_ret = build_composite_portfolio(factors, returns)


In [35]:
market_prices = get_price_data(["SPY"], "2015-01-01", "2025-01-01")
close_prices = market_prices["Close"]
spy_prices = close_prices["SPY"]
market_ret = compute_returns(spy_prices)

market_ret.head()


Date
2015-01-05   -0.018060
2015-01-06   -0.009419
2015-01-07    0.012461
2015-01-08    0.017745
2015-01-09   -0.008013
Name: SPY, dtype: float64

In [36]:
combined = pd.concat([comp_ret, market_ret], axis=1, keys=["comp", "SPY"]).dropna()
combined.head(), combined.shape


(            comp       SPY
 Date                      
 2015-01-05   0.0 -0.018060
 2015-01-06   0.0 -0.009419
 2015-01-07   0.0  0.012461
 2015-01-08   0.0  0.017745
 2015-01-09   0.0 -0.008013,
 (2515, 2))

In [37]:
from factor_engines.load_data import get_price_data, compute_returns
from factor_engines.regression import summarize_regression

market_prices = get_price_data(["SPY"], "2015-01-01", "2025-01-01")
close_prices = market_prices["Close"]
spy_prices = close_prices["SPY"]
#print(market_prices)
market_ret = compute_returns(spy_prices)

summary, results = summarize_regression(comp_ret, market_ret)

summary


{'alpha_daily': np.float64(0.022880456551518515),
 'alpha_annual': np.float64(5.7658750509826655),
 'alpha_tstat': np.float64(10.32438290859813),
 'r2': np.float64(0.008473701084548368),
 'beta_SPY': -0.924385053947449,
 'beta_SPY_tstats': -4.6342631971324995}